In [0]:
%run ../configs/log_configs

In [0]:
# ─────────────────────────────────────────────
# Cell 1: Imports
# ─────────────────────────────────────────────
import json
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable
from datetime import datetime

today = datetime.now()
year  = today.strftime("%Y")
month = today.strftime("%m")
day   = today.strftime("%d")

In [0]:
# ─────────────────────────────────────────────
# Cell 2: Widgets (ADF injected)
# ─────────────────────────────────────────────
dbutils.widgets.text("environment",        "dev")
dbutils.widgets.text("source_name",        "")
dbutils.widgets.text("raw_path",           "")
dbutils.widgets.text("bronze_table",       "")
dbutils.widgets.text("partition_col",      "")
dbutils.widgets.text("z_order_col",        "")
dbutils.widgets.text("primary_key",        "")
dbutils.widgets.text("is_daily",           "Y")
dbutils.widgets.text("notebook_name",      "")
dbutils.widgets.text("pipeline_name",      "")
dbutils.widgets.text("layer",              "bronze")

environment   = dbutils.widgets.get("environment")
source_name   = dbutils.widgets.get("source_name")
raw_path      = dbutils.widgets.get("raw_path")
bronze_table  = dbutils.widgets.get("bronze_table")
partition_col = dbutils.widgets.get("partition_col")
z_order_col   = dbutils.widgets.get("z_order_col")
primary_key   = dbutils.widgets.get("primary_key")
is_daily      = dbutils.widgets.get("is_daily")
notebook_name = dbutils.widgets.get("notebook_name")
pipeline_name = dbutils.widgets.get("pipeline_name")
layer         = dbutils.widgets.get("layer")

In [0]:
# ─────────────────────────────────────────────
# Cell 3: Schema Registry Loader + Builder
# Reads from ADLS — fully metadata driven
# No hardcoding anywhere
# ─────────────────────────────────────────────
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType,
    IntegerType, LongType,
    BooleanType, TimestampType
)

# ── Type mapping — JSON string → Spark type ──
TYPE_MAP = {
    "string":    StringType(),
    "double":    DoubleType(),
    "integer":   IntegerType(),
    "long":      LongType(),
    "boolean":   BooleanType(),
    "timestamp": TimestampType()
}

def load_schema_registry(environment):
    path = (
        f"abfss://{environment}@{storage_account}"
        f".dfs.core.windows.net/"
        f"bronze/configs/schemas/schema_registry.json"
    )
    raw = spark.read.text(path).collect()
    return json.loads(
        "\n".join([r.value for r in raw])
    )


def build_spark_schema(
    schema_registry: dict,
    source_name: str
) -> StructType:
    """
    Converts schema_registry JSON field list
    into a Spark StructType — no hardcoding.
    Falls back to inferSchema if source not found.
    """
    fields = schema_registry.get(source_name)

    if not fields:
        print(
            f"No schema found for '{source_name}' "
            f"in registry — will use inferSchema"
        )
        return None

    return StructType([
        StructField(
            f["name"],
            TYPE_MAP.get(f["type"], StringType()),
            True
        )
        for f in fields
    ])


def validate_schema(df, schema_registry_entry, source_name):
    """
    Validates actual columns against registry.
    schema_registry_entry is the list of field dicts.
    """
    actual_cols   = set(df.columns)
    expected_cols = set(f["name"] for f in schema_registry_entry)

    missing = expected_cols - actual_cols
    extra   = actual_cols - expected_cols

    if missing:
        raise ValueError(
            f"Schema drift detected for '{source_name}'. "
            f"Missing columns: {missing}"
        )
    if extra:
        print(
            f"Extra columns for '{source_name}': "
            f"{extra} — retained in bronze."
        )
    return True

In [0]:
# ─────────────────────────────────────────────
# Cell 4: File Existence Check
# ─────────────────────────────────────────────
def check_file_exists(path):
    try:
        files = dbutils.fs.ls(path)
        # Filter out Spark metadata files
        data_files = [
            f for f in files
            if not f.name.startswith("_")
            and not f.name.startswith(".")
        ]
        if not data_files:
            raise FileNotFoundError(
                f"No data files found at: {path}"
            )
        return True
    except Exception as e:
        raise FileNotFoundError(
            f"Path does not exist or is empty: {path}. "
            f"Error: {str(e)}"
        )

In [0]:
# ─────────────────────────────────────────────
# Cell 5: Bronze Writer
# Append + audit columns + Delta optimizations
# ─────────────────────────────────────────────
def write_bronze(df, bronze_table, partition_col,
                 z_order_col, source_name,
                 environment, layer):

    # ── Audit columns ──────────────────────
    df = (
        df
        .withColumn(
            "bib_ingested_at",
            F.current_timestamp()
        )
        .withColumn(
            "bib_source",
            F.lit(source_name)
        )
        .withColumn(
            "bib_environment",
            F.lit(environment)
        )
        .withColumn(
            "bib_is_deleted",
            F.lit(False)
        )
    )

    # ── Extract catalog/schema/table ───────
    parts  = bronze_table.split(".")
    cat    = parts[0]
    schema = parts[1]

    # ── Ensure schema exists ───────────────
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {cat}.{schema}"
    )

    # ── ADLS path ──────────────────────────
    adls_bronze_path = get_adls_path(
        environment, layer,
        f"{source_name}/"
    )

    # ── Partition + Z-order flags ──────────
    # is_valid_partition = (
    #     partition_col
    #     and partition_col.strip().upper() != "NA"
    # )
    # is_valid_zorder = (
    #     z_order_col
    #     and z_order_col.strip().upper() != "NA"
    # )

    # ── Write to Unity Catalog (Delta) ─────
    writer = (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
    )
    # if is_valid_partition:
    #     writer = writer.partitionBy(partition_col)

    writer.saveAsTable(bronze_table)
    print(f"Unity Catalog write done: {bronze_table}")

    # ── Write to ADLS (Delta) ──────────────
    adls_writer = (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
    )
    # if is_valid_partition:
    #     adls_writer = adls_writer.partitionBy(
    #         partition_col
    #     )

    adls_writer.save(adls_bronze_path)
    print(f"ADLS write done: {adls_bronze_path}")

    # ── OPTIMIZE + Z-ORDER ─────────────────
    # Always run — cheap for small tables
    # if is_valid_zorder:
    #     spark.sql(
    #         f"OPTIMIZE {bronze_table} "
    #         f"ZORDER BY ({z_order_col})"
    #     )
    # else:
    #     spark.sql(f"OPTIMIZE {bronze_table}")

    # print(f"OPTIMIZE done: {bronze_table}")

    # ── VACUUM (retain 7 days) ─────────────
    # spark.sql(
    #     f"VACUUM {bronze_table} RETAIN 168 HOURS"
    # )

    print(f"Bronze write complete: {bronze_table}")

In [0]:
try:
    # ── 1. Build raw path ──────────────────
    full_raw_path = get_raw_path(
        environment, raw_path,
        year, month, day, is_daily
    )
    print(f"Reading from: {full_raw_path}")

    # ── 2. Check file exists ───────────────
    check_file_exists(full_raw_path)

    # ── 3. Load schema registry once ───────
    schema_registry = load_schema_registry(
        environment
    )

    # ── 4. Build Spark schema from registry─
    # No hardcoding — driven by JSON config
    spark_schema = build_spark_schema(
        schema_registry, source_name
    )

    # ── 5. Read CSV ────────────────────────
    if spark_schema:
        df_raw = (
            spark.read
            .option("header", "true")
            .schema(spark_schema)     # fast — no inferSchema
            .csv(full_raw_path)
        )
    else:
        df_raw = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(full_raw_path)
        )

    # ── 6. Cache + count ───────────────────
    df_raw.cache()
    record_count = df_raw.count()
    print(f"Records read: {record_count}")

    # ── 7. Validate schema drift ───────────
    source_fields = schema_registry.get(source_name)
    if source_fields:
        validate_schema(
            df_raw, source_fields, source_name
        )

    # ── 8. Write bronze ────────────────────
    write_bronze(
        df_raw, bronze_table,
        partition_col, z_order_col,
        source_name, environment, layer
    )

    df_raw.unpersist()

    # ── 9. Log SUCCESS ─────────────────────
    log_pipeline_event(
        notebook_name=notebook_name,
        pipeline_name=pipeline_name,
        source_name=source_name,
        layer=layer,
        status="SUCCESS",
        records_processed=record_count,
        message=(
            f"{source_name} ingested to bronze. "
            f"Table: {bronze_table}"
        ),
        environment=environment
    )
    print(f"SUCCESS: {source_name} → {bronze_table}")

except FileNotFoundError as fe:
    log_pipeline_event(
        notebook_name=notebook_name,
        pipeline_name=pipeline_name,
        source_name=source_name,
        layer=layer,
        status="SKIPPED",
        records_processed=0,
        message=f"File not found: {str(fe)}",
        environment=environment
    )
    print(f"Skipped {source_name}: {str(fe)}")

except Exception as e:
    log_pipeline_event(
        notebook_name=notebook_name,
        pipeline_name=pipeline_name,
        source_name=source_name,
        layer=layer,
        status="FAILED",
        records_processed=0,
        message=f"Bronze load failed: {str(e)}",
        environment=environment
    )
    raise

In [0]:
# Run this once in a separate notebook cell
# then re-run the pipeline

# tables_to_reset = [
#     "bib_catalog.bronze.transactions",
#     "bib_catalog.bronze.digital_activity",
#     "bib_catalog.bronze.campaign_interactions",
#     "bib_catalog.bronze.customers",
#     "bib_catalog.bronze.accounts",
#     "bib_catalog.bronze.product_holdings",
#     "bib_catalog.bronze.products",
#     "bib_catalog.bronze.branches",
#     "bib_catalog.bronze.bank_managers"
# ]

# for table in tables_to_reset:
#     spark.sql(f"DROP TABLE IF EXISTS {table}")
#     print(f"Dropped: {table}")

In [0]:
# import re

# bronze_sources = [
#     "transactions", "digital_activity",
#     "campaign_interactions", "customers",
#     "accounts", "product_holdings",
#     "products", "branches", "bank_managers"
# ]

# for source in bronze_sources:
#     path = (
#         f"abfss://dev@{storage_account}"
#         f".dfs.core.windows.net/bronze/{source}/"
#     )
#     try:
#         dbutils.fs.rm(path, recurse=True)
#         print(f"Cleaned: {path}")
#     except:
#         print(f"Not found (skip): {path}")